# Creating a Simple Agent with Tracing

In [24]:
import os
import dotenv
from openai import OpenAI

# Garante que as variáveis de ambiente estão atualizadas
dotenv.load_dotenv(override=True)

os.environ["OPENAI_API_KEY"] = os.environ.get("GROQ_API_KEY")
os.environ["OPENAI_BASE_URL"] = "https://api.groq.com/openai/v1"

# Motor da Groq
client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

# A adaptação para o padrão universal
response = client.chat.completions.create(
    model=os.environ.get("OPENAI_DEFAULT_MODEL"), # Puxa o Llama 3 do seu .env
    messages=[
        {"role": "user", "content": "Say: We are up and running!"}
    ]
)

# Imprimindo o resultado
print(response.choices[0].message.content)

We are up and running!


In [25]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [26]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions='''
You are a helpful assistant giving out nutrition advice,
You give concise answers.
'''
)

Let's execute the Agent:

In [27]:
with trace("Simple Nutrition Agent"): #trace - rastreamento de execução do agente
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

OPENAI_API_KEY is not set, skipping trace export


RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Bananas are a nutrient-rich fruit, high in potassium, vitamins C and B6, and fiber. They support healthy digestion, heart function, and energy production. One medium banana provides about 105 calories and 3 grams of fiber. Overall, bananas are a healthy and nutritious addition to a balanced diet.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [28]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)
    print()






Ban
anas
 are
 a
 nutritious
 fruit
,
 rich
 in
 fiber
,
 vitamins
 C
 and
 B
6
,
 and
 potassium
.
 They
 support
 healthy
 digestion
,
 heart
 function
,
 and
 can
 help
 lower
 blood
 pressure
.
 One
 medium
 banana
 provides
 about
 
105
 calories
 and
 
3
 grams
 of
 fiber
.
 They
're
 a
 great
,
 healthy
 snack
.










OPENAI_API_KEY is not set, skipping trace export


_Good Job!_